In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

In [ ]:
data_dir = "dataset_ropa"
labels_file = "dataset_ropa/labels.csv"
# Que opcion tomar, label en CSV o carpeta con subcarpetas

categorias = ["Tiki", "Casi Tiki", "Descarte"] # Categorias a clasificar ****
defectos = ["Manchas", "Agujeros", "Motitas", "Decoloración"] # defectos a clasificar

In [ ]:
# Cargado de labels
df = pd.read_csv(labels_file)
df["categoria"] = df["categoria"].astype(str)
df["defectos"] = df["defectos"].fillna("")

In [ ]:
# Parámetros
target_size = (224, 224)
batch_size = 32

In [ ]:
# Preprocesamiento y aumento de datos en tiempo real
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

In [ ]:
# Generadores de datos a partir del DataFrame
train_generator = train_datagen.flow_from_dataframe(
    dataframe=df,
    directory=data_dir,
    x_col="filename",
    y_col="categoria",
    target_size=target_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_dataframe(
    dataframe=df,
    directory=data_dir,
    x_col="filename",
    y_col="categoria",
    target_size=target_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = True 

for layer in base_model.layers[:100]:
    layer.trainable = False

In [ ]:
x = base_model.output
x = GlobalAveragePoolinx = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
categoria_output = Dense(len(categorias), activation='softmax', name='categoria_output')  # Clasificación de estado
defectos_output = Dense(len(defectos), activation='sigmoid', name='defectos_output')  # Clasificación de defectos

model = Model(inputs=base_model.input, outputs=[categoria_output(x), defectos_output(x)])


In [ ]:
model.compile(optimizer='adam', 
            loss={'categoria_output': 'categorical_crossentropy', 'defectos_output': 'binary_crossentropy'},
            metrics=['accuracy'])
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

epochs = 30
model.fit(train_generator, validation_data=val_generator, epochs=epochs, callbacks=[early_stopping])

model.save("modelo_ropa_multi.keras")
print("Modelo entrenado y guardado correctamente.")


In [ ]:
y_true = val_generator.classes
y_pred_categoria, y_pred_defectos = model.predict(val_generator)
y_pred_categoria = np.argmax(y_pred_categoria, axis=1)

y_true_defectos = np.array([[1 if defecto in str(row) else 0 for defecto in defectos] for row in df["defectos"]])
y_pred_defectos = (y_pred_defectos > 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred_categoria)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=categorias, yticklabels=categorias)
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de Confusión - Categoría")
plt.show()

print("Reporte de Clasificación - Categoría")
print(classification_report(y_true, y_pred_categoria, target_names=categorias))

print("Reporte de Clasificación - Defectos")
for i, defecto in enumerate(defectos):
    print(f"{defecto}:")
    print(classification_report(y_true_defectos[:, i], y_pred_defectos[:, i]))


# Ver opciones de que forma seria mejor:
## - Hacer calculos con total de la integridad de la prenda, ya sea por ejemplo, análisis de la prenda con valores de 0 a 1, hablando en % 0 - 25% Descarte, mayor a 25% hasta 65% Casi tiki y mayor a 65% hasta 95% Tiki. Por lo tanto, el modelo entrega un valor solo definiendo la integridad, y posteriormente se realizan los calculos dependiendo si tiene defectos y que cada defecto tenga su peso reduciendo la integridad de la prenda. 